# Ablation 2, Variant E: Attention Layer Ratio Sweep (Attention Ratio Sweep Results

**The Goal**: Titrate the "dose" of attention within StripedHyena from 0%
(pure Hyena, no attention blocks) through 25%, 50%, 75%, to 100% (pure
attention, equivalent to a Transformer). Plot composite stability vs.
attention fraction to produce a **dose-response curve** for the geometric tax.

## Design

- Use **8-layer** StripedHyena variants to allow fine-grained ratio control
- Sweep attention fractions: **0/8, 1/8, 2/8, 3/8, 4/8, 5/8, 6/8, 7/8, 8/8**
- All variants trained with identical hyperparameters and discrete CE loss
- Evaluation via Shesha Procrustes stability harness (identical to baseline)

### Hypotheses

**H1 (Linear)**: Each attention layer independently contributes a fixed
quantum of geometric tax. Stability degrades linearly with attention ratio.

**H2 (Phase Transition)**: The manifold tolerates a limited fraction of
attention before shattering. There exists a critical threshold above which
stability collapses.

**H3 (Diminishing Returns)**: The first few attention layers cause most of
the damage; additional attention layers contribute less marginal tax.

**Expected Result**: Monotonic decrease in composite stability as attention
fraction increases, with SmallBERT and SmallMamba baselines as reference
bounds. The shape of the curve (linear vs. sigmoidal vs. concave) reveals
the nature of how attention disrupts continuous manifold geometry.

## Prerequisites
- Upload `evaluation_harness.py` to `/content/`
- GPU runtime required
- Run after baseline `Architectural_Control_Experiment.ipynb`

---

In [ ]:
# Install Dependencies

print('Installing core dependencies...')
!pip install -q torch shesha-geometry matplotlib seaborn pandas scipy

print('\nBuilding mamba-ssm from source...')
!pip install causal-conv1d mamba-ssm --no-build-isolation 2>&1 | tail -5

MAMBA_AVAILABLE = False
try:
    from mamba_ssm import Mamba
    MAMBA_AVAILABLE = True
    print('mamba-ssm: OK (native CUDA)')
except ImportError:
    print('mamba-ssm: FAILED -- will use pure-PyTorch SSM fallback')

import torch
print(f'\ntorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print('Done!')

In [ ]:
# Configuration

import os, sys, gc, time
import numpy as np
import pandas as pd
sys.path.insert(0, '.')

OUTPUT_BASE = './results/ablation2_variant_e_attn_ratio/'
RESULTS_DIR = OUTPUT_BASE + 'results'
CACHE_DIR   = OUTPUT_BASE + 'cache'
CKPT_DIR    = OUTPUT_BASE + 'checkpoints'
for d in [RESULTS_DIR, CACHE_DIR, CKPT_DIR]:
    os.makedirs(d, exist_ok=True)

SEQ_LEN    = 512
N_BINS     = 256
N_TRAIN    = 50_000
N_EVAL     = 2_000
SEED       = 320
PAD_TOKEN  = N_BINS
MASK_TOKEN = N_BINS + 1
VOCAB_SIZE = N_BINS + 2

# 8 layers to allow fine-grained ratio control (9 sweep points)
D_MODEL    = 256
N_LAYERS   = 8
N_HEADS    = 4
FFN_DIM    = 1024
D_STATE    = 16
D_CONV     = 4
EXPAND     = 2

LR         = 3e-4
WEIGHT_DECAY = 0.01
EPOCHS     = 20
BATCH_SIZE = 64
NOISE_RATES = [0.01, 0.02, 0.05, 0.10]
DATASETS = ['waveform', 'oscillator', 'lorenz']

# Attention ratio sweep: 0/8 through 8/8
# Each entry is a list of layer indices that use attention (rest use Hyena)
ATTENTION_CONFIGS = {
    '0/8 (pure Hyena)':   [],
    '1/8 (12.5%)':        [4],
    '2/8 (25.0%)':        [3, 7],
    '3/8 (37.5%)':        [2, 5, 7],
    '4/8 (50.0%)':        [1, 3, 5, 7],
    '5/8 (62.5%)':        [1, 2, 4, 6, 7],
    '6/8 (75.0%)':        [0, 1, 3, 5, 6, 7],
    '7/8 (87.5%)':        [0, 1, 2, 3, 5, 6, 7],
    '8/8 (pure Attn)':    [0, 1, 2, 3, 4, 5, 6, 7],
}

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'Ablation: Variant E -- Attention Layer Ratio Sweep')
print(f'Architecture: 8-layer StripedHyena with variable attention ratio')
print(f'Sweep: {len(ATTENTION_CONFIGS)} configurations ({N_LAYERS} layers each)')
print(f'Output: {OUTPUT_BASE}')

In [ ]:
# Dataset Generators (FIX v2: global discretization)

from scipy.integrate import solve_ivp

def discretize(values, n_bins=N_BINS, global_min=None, global_max=None):
    '''Uses dataset-global min/max when provided.'''
    vmin = global_min if global_min is not None else values.min()
    vmax = global_max if global_max is not None else values.max()
    if vmax - vmin < 1e-12:
        return np.full_like(values, n_bins // 2, dtype=np.int64)
    normed = np.clip((values - vmin) / (vmax - vmin), 0.0, 1.0)
    return np.clip((normed * (n_bins - 1)).astype(np.int64), 0, n_bins - 1)

def generate_waveforms(n_sequences, seq_len=SEQ_LEN, seed=SEED, global_range=None):
    rng = np.random.default_rng(seed)
    t = np.linspace(0, 1, seq_len, endpoint=False)
    raw = []
    for _ in range(n_sequences):
        n_sines = rng.integers(3, 6)
        signal = np.zeros(seq_len)
        for _ in range(n_sines):
            signal += rng.uniform(0.1, 1.0) * np.sin(
                2*np.pi * rng.uniform(0.5, 50.0) * t + rng.uniform(0, 2*np.pi))
        raw.append(signal)
    if global_range is None:
        all_v = np.concatenate(raw)
        global_range = (float(all_v.min()), float(all_v.max()))
    gmin, gmax = global_range
    seqs = [discretize(s, global_min=gmin, global_max=gmax) for s in raw]
    return np.array(seqs, dtype=np.int64), global_range

def generate_oscillators(n_sequences, seq_len=SEQ_LEN, seed=SEED, global_range=None):
    rng = np.random.default_rng(seed)
    t = np.linspace(0, 4, seq_len, endpoint=False)
    raw = []
    for _ in range(n_sequences):
        signal = rng.uniform(0.5,2)*np.exp(-rng.uniform(0.2,2)*t)*np.cos(
            rng.uniform(2,20)*t + rng.uniform(0,2*np.pi))
        raw.append(signal)
    if global_range is None:
        all_v = np.concatenate(raw)
        global_range = (float(all_v.min()), float(all_v.max()))
    gmin, gmax = global_range
    seqs = [discretize(s, global_min=gmin, global_max=gmax) for s in raw]
    return np.array(seqs, dtype=np.int64), global_range

def _lorenz_rhs(t, state, sigma=10.0, rho=28.0, beta=8.0/3.0):
    x, y, z = state
    return [sigma*(y-x), x*(rho-z)-y, x*y-beta*z]

def generate_lorenz(n_sequences, seq_len=SEQ_LEN, seed=SEED, global_range=None):
    rng = np.random.default_rng(seed)
    t_span = (0, 25)
    t_eval = np.linspace(*t_span, seq_len)
    raw = []
    for _ in range(n_sequences):
        x0 = rng.uniform(-15, 15)
        y0 = rng.uniform(-15, 15)
        z0 = rng.uniform(10, 40)
        sol = solve_ivp(_lorenz_rhs, t_span, [x0, y0, z0], t_eval=t_eval,
                        method='RK45', max_step=0.05)
        if sol.success and len(sol.y[0]) == seq_len:
            raw.append(sol.y[0])
        else:
            sol2 = solve_ivp(_lorenz_rhs, t_span,
                             [x0 + rng.uniform(-1, 1), y0, z0],
                             t_eval=t_eval, method='RK45', max_step=0.01)
            sig = sol2.y[0][:seq_len]
            if len(sig) < seq_len:
                sig = np.pad(sig, (0, seq_len - len(sig)), mode='edge')
            raw.append(sig)
    if global_range is None:
        all_v = np.concatenate(raw)
        global_range = (float(all_v.min()), float(all_v.max()))
    gmin, gmax = global_range
    seqs = [discretize(s, global_min=gmin, global_max=gmax) for s in raw]
    return np.array(seqs, dtype=np.int64), global_range

GENERATORS = {'waveform': generate_waveforms, 'oscillator': generate_oscillators,
              'lorenz': generate_lorenz}
GLOBAL_RANGES = {}
print('Dataset generators ready (3 continuous dynamical systems)')

In [ ]:
# Perturbation Suite (identical to baseline)

from dataclasses import dataclass, field

@dataclass
class PerturbedSet:
    name: str
    category: str
    sequences: np.ndarray
    params: dict = field(default_factory=dict)
    description: str = ''

def noise_perturb(sequences, rate, rng, n_bins=N_BINS):
    out = sequences.copy()
    mask = rng.random(out.shape) < rate
    noise = rng.normal(0, n_bins * 0.10, size=out.shape).astype(np.int64)
    out[mask] = np.clip(out[mask] + noise[mask], 0, n_bins - 1)
    return out

class ContinuousPerturbationSuite:
    def __init__(self, seed=SEED, noise_rates=None):
        self.rng = np.random.default_rng(seed)
        self.noise_rates = noise_rates or NOISE_RATES
    def run_all(self, sequences):
        results = {}
        for rate in self.noise_rates:
            name = f'noise_{int(rate*100)}pct'
            results[name] = PerturbedSet(name=name, category='noise',
                sequences=noise_perturb(sequences, rate, self.rng),
                params={'noise_rate': rate})
        results['time_reversal'] = PerturbedSet(name='time_reversal', category='reversal',
            sequences=sequences[:, ::-1].copy())
        return results

print('ContinuousPerturbationSuite ready')

In [ ]:
# Model Definitions -- StripedHyena with configurable attention ratio
#
# Reuses the StripedHyena building blocks (ImplicitFilterMLP, HyenaOperator,
# SHMultiHeadAttention, StripedHyenaBlock) from Variant B, with the key
# difference that attention_layers is explicitly swept rather than fixed.
#
# Also includes SmallBERT and SmallMamba as reference baselines.

import torch
import torch.nn as nn
import torch.nn.functional as F
import math


# =====================================================================
# StripedHyena components (identical to Variant B)
# =====================================================================

class ImplicitFilterMLP(nn.Module):
    """Learns long convolution filter implicitly via MLP (Hyena's key innovation)."""
    def __init__(self, d_model, seq_len, n_hidden=64):
        super().__init__()
        self.d_model = d_model
        self.seq_len = seq_len
        n_pos_features = 16
        self.pos_emb = nn.Linear(n_pos_features, n_hidden)
        self.mlp = nn.Sequential(
            nn.GELU(), nn.Linear(n_hidden, n_hidden),
            nn.GELU(), nn.Linear(n_hidden, d_model))
        self.decay = nn.Parameter(torch.linspace(0.01, 5.0, d_model))
        self.register_buffer('pos_features',
                             self._make_pos_features(seq_len, n_pos_features))

    def _make_pos_features(self, seq_len, n_features):
        positions = torch.linspace(0, 1, seq_len).unsqueeze(1)
        freqs = torch.arange(n_features).float() * math.pi
        return torch.sin(positions * freqs.unsqueeze(0))

    def forward(self, seq_len):
        if seq_len != self.seq_len:
            pf = self._make_pos_features(
                seq_len, self.pos_features.shape[1]).to(self.pos_features.device)
        else:
            pf = self.pos_features
        h = self.mlp(self.pos_emb(pf))
        t = torch.linspace(0, 1, seq_len, device=h.device).unsqueeze(1)
        return (h * torch.exp(-self.decay.unsqueeze(0) * t)).T


class HyenaOperator(nn.Module):
    """Data-controlled long convolution with gating (replaces attention)."""
    def __init__(self, d_model, seq_len, order=2, short_conv_kernel=3):
        super().__init__()
        self.d_model, self.order = d_model, order
        self.in_proj = nn.Linear(d_model, (order + 1) * d_model)
        self.short_convs = nn.ModuleList([
            nn.Conv1d(d_model, d_model, kernel_size=short_conv_kernel,
                      padding=short_conv_kernel // 2, groups=d_model)
            for _ in range(order + 1)])
        self.filters = nn.ModuleList([
            ImplicitFilterMLP(d_model, seq_len) for _ in range(order)])
        self.out_proj = nn.Linear(d_model, d_model)

    def _fft_conv(self, signal, kernel):
        L = signal.shape[-1]
        fft_len = 2 * L
        return torch.fft.irfft(
            torch.fft.rfft(signal, n=fft_len, dim=-1) *
            torch.fft.rfft(kernel, n=fft_len, dim=-1).unsqueeze(0),
            n=fft_len, dim=-1)[..., :L]

    def forward(self, x):
        B, L, D = x.shape
        branches = self.in_proj(x).reshape(B, L, self.order + 1, D)
        conv_outputs = [
            self.short_convs[i](branches[:, :, i, :].transpose(1, 2))
            for i in range(self.order + 1)]
        v = conv_outputs[0]
        for i in range(self.order):
            v = self._fft_conv(v, self.filters[i](L)) * conv_outputs[i + 1]
        return self.out_proj(v.transpose(1, 2))


class SHMultiHeadAttention(nn.Module):
    """Standard multi-head attention with RoPE (for StripedHyena attention layers)."""
    def __init__(self, d_model, n_heads, max_seq_len=2048):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.qkv_proj = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        freqs = 1.0 / (10000 ** (
            torch.arange(0, self.head_dim, 2).float() / self.head_dim))
        self.register_buffer('freqs',
            torch.outer(torch.arange(max_seq_len).float(), freqs))

    def _apply_rotary(self, x, freqs):
        L = x.shape[2]
        freqs = freqs[:L]
        cos_f = torch.cos(freqs).unsqueeze(0).unsqueeze(0)
        sin_f = torch.sin(freqs).unsqueeze(0).unsqueeze(0)
        x1, x2 = x[..., ::2], x[..., 1::2]
        return torch.stack(
            [x1 * cos_f - x2 * sin_f, x1 * sin_f + x2 * cos_f],
            dim=-1).flatten(-2)

    def forward(self, x):
        B, L, D = x.shape
        qkv = self.qkv_proj(x).reshape(B, L, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.permute(2, 0, 3, 1, 4)
        q = self._apply_rotary(q, self.freqs)
        k = self._apply_rotary(k, self.freqs)
        attn = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        mask = torch.triu(torch.ones(L, L, device=x.device, dtype=torch.bool),
                          diagonal=1)
        attn = F.softmax(
            attn.masked_fill(mask.unsqueeze(0).unsqueeze(0), float('-inf')),
            dim=-1)
        return self.out_proj((attn @ v).transpose(1, 2).reshape(B, L, D))


class StripedHyenaBlock(nn.Module):
    """Single StripedHyena block: Hyena or Attention + SwiGLU MLP."""
    def __init__(self, d_model, seq_len, n_heads=4, order=2,
                 is_attention=False, mlp_ratio=4):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        if is_attention:
            self.mixer = SHMultiHeadAttention(d_model, n_heads)
        else:
            self.mixer = HyenaOperator(d_model, seq_len, order=order)
        mlp_hidden = int(d_model * mlp_ratio)
        self.mlp_gate = nn.Linear(d_model, mlp_hidden)
        self.mlp_value = nn.Linear(d_model, mlp_hidden)
        self.mlp_out = nn.Linear(mlp_hidden, d_model)

    def forward(self, x):
        x = x + self.mixer(self.norm1(x))
        h = self.norm2(x)
        x = x + self.mlp_out(F.silu(self.mlp_gate(h)) * self.mlp_value(h))
        return x


class SmallStripedHyena(nn.Module):
    """StripedHyena with configurable attention layer placement.

    The attention_layers parameter controls WHICH layers use attention
    (softmax) vs. Hyena (long convolution). This is the sole independent
    variable in the ratio sweep.
    """
    def __init__(self, vocab_size=VOCAB_SIZE, d_model=D_MODEL,
                 n_layers=N_LAYERS, n_heads=N_HEADS, seq_len=SEQ_LEN,
                 order=2, mlp_ratio=4, attention_layers=None, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.vocab_size = vocab_size
        if attention_layers is None:
            attention_layers = []
        self.attention_layers = set(attention_layers)
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([
            StripedHyenaBlock(
                d_model, seq_len, n_heads, order,
                is_attention=(i in self.attention_layers),
                mlp_ratio=mlp_ratio)
            for i in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self._init_weights()
        n_p = sum(p.numel() for p in self.parameters()) / 1e6
        n_attn = len(self.attention_layers)
        print(f'SmallStripedHyena: {n_p:.2f}M params '
              f'({n_layers - n_attn} Hyena + {n_attn} Attn)')

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.normal_(p, 0, 0.02)

    def forward(self, x, return_hidden=False):
        h = self.tok_emb(x)
        for block in self.blocks:
            h = block(h)
        h = self.norm(h)
        logits = self.head(h)
        return (logits, h) if return_hidden else logits


# =====================================================================
# Reference baselines: SmallBERT (8-layer) and SmallMamba (8-layer)
# =====================================================================

class SmallBERT(nn.Module):
    """8-layer Transformer baseline (matches sweep layer count)."""
    def __init__(self, vocab_size=VOCAB_SIZE, d_model=D_MODEL, nhead=N_HEADS,
                 num_layers=N_LAYERS, dim_feedforward=FFN_DIM,
                 max_seq_len=SEQ_LEN, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        self.drop = nn.Dropout(dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(
            encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)
        self._init_weights()
        n_p = sum(p.numel() for p in self.parameters()) / 1e6
        print(f'SmallBERT: {n_p:.2f}M params ({num_layers} layers)')

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def _causal_mask(self, seq_len, device):
        mask = torch.triu(torch.ones(seq_len, seq_len, device=device),
                          diagonal=1)
        return mask.masked_fill(mask == 1, float('-inf'))

    def forward(self, x, return_hidden=False):
        B, L = x.shape
        pos = torch.arange(L, device=x.device).unsqueeze(0)
        h = self.drop(self.tok_emb(x) + self.pos_emb(pos))
        h = self.encoder(h, mask=self._causal_mask(L, x.device))
        h = self.norm(h)
        logits = self.head(h)
        return (logits, h) if return_hidden else logits


if MAMBA_AVAILABLE:
    from mamba_ssm import Mamba as MambaBlock

    class SmallMamba(nn.Module):
        """8-layer Mamba baseline (matches sweep layer count)."""
        def __init__(self, vocab_size=VOCAB_SIZE, d_model=D_MODEL,
                     num_layers=N_LAYERS, d_state=D_STATE, d_conv=D_CONV,
                     expand=EXPAND, max_seq_len=SEQ_LEN, dropout=0.1):
            super().__init__()
            self.d_model = d_model
            self.tok_emb = nn.Embedding(vocab_size, d_model)
            self.layers = nn.ModuleList()
            self.norms = nn.ModuleList()
            for _ in range(num_layers):
                self.layers.append(MambaBlock(
                    d_model=d_model, d_state=d_state,
                    d_conv=d_conv, expand=expand))
                self.norms.append(nn.LayerNorm(d_model))
            self.final_norm = nn.LayerNorm(d_model)
            self.head = nn.Linear(d_model, vocab_size, bias=False)
            self._init_weights()
            n_p = sum(p.numel() for p in self.parameters()) / 1e6
            print(f'SmallMamba: {n_p:.2f}M params ({num_layers} layers)')

        def _init_weights(self):
            for p in self.parameters():
                if p.dim() > 1:
                    nn.init.normal_(p, 0, 0.02)

        def forward(self, x, return_hidden=False):
            h = self.tok_emb(x)
            for layer, norm in zip(self.layers, self.norms):
                h = h + layer(norm(h))
            h = self.final_norm(h)
            logits = self.head(h)
            return (logits, h) if return_hidden else logits
else:
    # Pure-PyTorch SSM fallback
    class SimpleSSSM(nn.Module):
        def __init__(self, d_model, d_state=16):
            super().__init__()
            self.d_model, self.d_state = d_model, d_state
            self.A = nn.Parameter(torch.randn(d_model, d_state) * 0.01)
            self.B_proj = nn.Linear(d_model, d_state)
            self.C_proj = nn.Linear(d_model, d_state)
            self.D = nn.Parameter(torch.ones(d_model))
            self.dt_proj = nn.Linear(d_model, d_model)
            self.in_proj = nn.Linear(d_model, d_model * 2)
            self.out_proj = nn.Linear(d_model, d_model)
        def forward(self, x):
            B_sz, L, D = x.shape
            xz = self.in_proj(x)
            x_in, z = xz.chunk(2, dim=-1)
            dt = F.softplus(self.dt_proj(x_in))
            B_mat = self.B_proj(x_in)
            C_mat = self.C_proj(x_in)
            A_bar = torch.exp(self.A.unsqueeze(0).unsqueeze(0) * dt.unsqueeze(-1))
            B_bar = dt.unsqueeze(-1) * B_mat.unsqueeze(2)
            h = torch.zeros(B_sz, D, self.d_state, device=x.device)
            ys = []
            for t in range(L):
                h = A_bar[:, t] * h + B_bar[:, t] * x_in[:, t].unsqueeze(-1)
                y = (h * C_mat[:, t].unsqueeze(1)).sum(-1)
                ys.append(y)
            y = torch.stack(ys, dim=1)
            return self.out_proj(y * F.silu(z)) + x * self.D.unsqueeze(0).unsqueeze(0)

    class SmallMamba(nn.Module):
        def __init__(self, vocab_size=VOCAB_SIZE, d_model=D_MODEL,
                     num_layers=N_LAYERS, d_state=D_STATE, d_conv=D_CONV,
                     expand=EXPAND, max_seq_len=SEQ_LEN, dropout=0.1):
            super().__init__()
            self.d_model = d_model
            self.tok_emb = nn.Embedding(vocab_size, d_model)
            self.layers = nn.ModuleList()
            self.norms = nn.ModuleList()
            for _ in range(num_layers):
                self.layers.append(SimpleSSSM(d_model, d_state))
                self.norms.append(nn.LayerNorm(d_model))
            self.final_norm = nn.LayerNorm(d_model)
            self.head = nn.Linear(d_model, vocab_size, bias=False)
            self._init_weights()
            n_p = sum(p.numel() for p in self.parameters()) / 1e6
            print(f'SmallMamba (fallback): {n_p:.2f}M params ({num_layers} layers)')
        def _init_weights(self):
            for p in self.parameters():
                if p.dim() > 1: nn.init.normal_(p, 0, 0.02)
        def forward(self, x, return_hidden=False):
            h = self.tok_emb(x)
            for layer, norm in zip(self.layers, self.norms):
                h = h + layer(norm(h))
            h = self.final_norm(h)
            logits = self.head(h)
            return (logits, h) if return_hidden else logits


# Verify model construction for boundary configs
m0 = SmallStripedHyena(attention_layers=[])
m8 = SmallStripedHyena(attention_layers=list(range(8)))
print(f'Pure Hyena (0/8): {sum(p.numel() for p in m0.parameters())/1e6:.2f}M')
print(f'Pure Attn  (8/8): {sum(p.numel() for p in m8.parameters())/1e6:.2f}M')
del m0, m8
print('All models ready')

In [ ]:
# Training Loop (CLM with CrossEntropy)
#
# Standard training identical to the main experiment. No Jacobian penalty,
# no continuous head. The ONLY variable is the attention layer placement.

from torch.utils.data import DataLoader, TensorDataset

def train_model(model, train_data, val_data, arch_name, ds_name,
                epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, seed=SEED):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    model = model.to(DEVICE)
    tl = DataLoader(TensorDataset(torch.from_numpy(train_data).long()),
                    batch_size=batch_size, shuffle=True, drop_last=True,
                    num_workers=2, pin_memory=True)
    vl = DataLoader(TensorDataset(torch.from_numpy(val_data).long()),
                    batch_size=batch_size, shuffle=False,
                    num_workers=2, pin_memory=True)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt, T_max=epochs * len(tl))
    criterion = nn.CrossEntropyLoss()

    best_val, best_state = float('inf'), None
    ckpt = f'{CKPT_DIR}/{arch_name}_{ds_name}_seed{seed}_best.pt'

    if os.path.exists(ckpt):
        model.load_state_dict(torch.load(ckpt, map_location=DEVICE,
                                          weights_only=True))
        # Re-evaluate val CE
        model.eval()
        vloss, vb = 0, 0
        with torch.no_grad():
            for (bx,) in vl:
                bx = bx.to(DEVICE)
                inp, tgt = bx[:, :-1], bx[:, 1:]
                vloss += criterion(
                    model(inp).reshape(-1, VOCAB_SIZE),
                    tgt.reshape(-1)).item()
                vb += 1
        loaded_ce = vloss / max(vb, 1)
        print(f'  Loaded: {ckpt} (val CE: {loaded_ce:.4f})')
        return model, [], [], loaded_ce

    print(f'  Training {arch_name} on {ds_name} ({epochs} epochs)...')
    t0 = time.time()
    train_losses, val_losses = [], []

    for ep in range(epochs):
        model.train()
        eloss, nb = 0, 0
        for (bx,) in tl:
            bx = bx.to(DEVICE)
            inp, tgt = bx[:, :-1], bx[:, 1:]
            logits = model(inp)
            loss = criterion(logits.reshape(-1, VOCAB_SIZE), tgt.reshape(-1))
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            sched.step()
            eloss += loss.item()
            nb += 1
        train_losses.append(eloss / nb)

        model.eval()
        vloss, vb = 0, 0
        with torch.no_grad():
            for (bx,) in vl:
                bx = bx.to(DEVICE)
                inp, tgt = bx[:, :-1], bx[:, 1:]
                vloss += criterion(
                    model(inp).reshape(-1, VOCAB_SIZE),
                    tgt.reshape(-1)).item()
                vb += 1
        avg_v = vloss / max(vb, 1)
        val_losses.append(avg_v)

        if avg_v < best_val:
            best_val = avg_v
            best_state = {k: v.cpu().clone()
                          for k, v in model.state_dict().items()}
        if (ep + 1) % 5 == 0 or ep == 0:
            print(f'    Ep {ep+1:3d}/{epochs} '
                  f'train={eloss/nb:.4f} val={avg_v:.4f} '
                  f'[{time.time()-t0:.0f}s]')

    if best_state:
        model.load_state_dict(best_state)
    torch.save(model.state_dict(), ckpt)
    print(f'  Done in {(time.time()-t0)/60:.1f}min (best val: {best_val:.4f})')
    return model, train_losses, val_losses, best_val

print('Training loop ready (CLM + CrossEntropy)')

In [ ]:
# Evaluation Harness

from evaluation_harness import StabilityHarness

harness = StabilityHarness(
    window_size=0, metric='cosine', n_splits=30,
    seed=320, max_samples=2500, n_bootstrap=5,
)
print('Shesha StabilityHarness configured (bootstrap=5)')

In [ ]:
# Run Attention Ratio Sweep + Baselines
#
# For each attention configuration, train on all 3 datasets and evaluate
# geometric stability. Also train SmallBERT and SmallMamba as reference
# baselines (both at 8 layers to match the sweep architecture depth).

def cleanup_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

@torch.no_grad()
def extract_embeddings(model, sequences, batch_size=128):
    model.eval()
    all_embs = []
    for i in range(0, len(sequences), batch_size):
        batch = torch.from_numpy(
            sequences[i:i+batch_size]).long().to(DEVICE)
        _, hidden = model(batch, return_hidden=True)
        emb = hidden.mean(dim=1)
        all_embs.append(emb.cpu().numpy())
    return np.concatenate(all_embs, axis=0)


print('=' * 70)
print('ABLATION VARIANT E: ATTENTION LAYER RATIO SWEEP')
print(f'  Architecture: {N_LAYERS}-layer StripedHyena')
print(f'  Configs: {len(ATTENTION_CONFIGS)} attention ratios')
print(f'  + SmallBERT + SmallMamba baselines (both {N_LAYERS} layers)')
print('=' * 70)

all_rows = []

# --- Run the StripedHyena sweep ---
for config_name, attn_layers in ATTENTION_CONFIGS.items():
    n_attn = len(attn_layers)
    attn_frac = n_attn / N_LAYERS
    arch_tag = f'SH_attn{n_attn}of{N_LAYERS}'

    print(f"\n{'#' * 70}")
    print(f'CONFIG: {config_name}')
    print(f'  Attention layers: {attn_layers}')
    print(f'  Attention fraction: {attn_frac:.3f}')
    print('#' * 70)

    for ds_name in DATASETS:
        gen_fn = GENERATORS[ds_name]
        print(f"\n  {'=' * 60}")
        print(f'  DATASET: {ds_name.upper()} | {config_name}')
        print('  ' + '=' * 60)

        train_data, grange = gen_fn(N_TRAIN, seed=SEED)
        GLOBAL_RANGES[ds_name] = grange
        eval_data, _ = gen_fn(N_EVAL, seed=SEED + 1000,
                              global_range=grange)
        pert_suite = ContinuousPerturbationSuite(seed=SEED)
        perturbed_sets = pert_suite.run_all(eval_data)

        model = SmallStripedHyena(attention_layers=attn_layers)
        model, t_losses, v_losses, best_val_ce = train_model(
            model, train_data, eval_data, arch_tag, ds_name, seed=SEED)

        # Extract embeddings
        ref_embs = extract_embeddings(model, eval_data)
        for pert_name, pset in perturbed_sets.items():
            pert_embs = extract_embeddings(model, pset.sequences)
            report = harness.evaluate(arch_tag, ref_embs, pert_embs, pert_name)
            row = {
                'architecture': arch_tag,
                'config_name': config_name,
                'n_attn_layers': n_attn,
                'attn_fraction': attn_frac,
                'dataset': ds_name,
                'perturbation': pert_name,
                'rdm_similarity': report.rdm_similarity_score,
                'perturbation_stability': report.perturbation_stability_score,
                'composite': report.composite_stability,
                'val_ce': best_val_ce,
                'model_type': 'striped_hyena',
            }
            all_rows.append(row)
            print(f'    {pert_name:20s} composite={report.composite_stability:.4f}')

        del model
        cleanup_gpu()

# --- Run SmallBERT baseline ---
print(f"\n{'#' * 70}")
print('BASELINE: SmallBERT (pure Transformer)')
print('#' * 70)

for ds_name in DATASETS:
    gen_fn = GENERATORS[ds_name]
    train_data, grange = gen_fn(N_TRAIN, seed=SEED)
    eval_data, _ = gen_fn(N_EVAL, seed=SEED + 1000, global_range=grange)
    pert_suite = ContinuousPerturbationSuite(seed=SEED)
    perturbed_sets = pert_suite.run_all(eval_data)

    model = SmallBERT()
    model, t_losses, v_losses, best_val_ce = train_model(
        model, train_data, eval_data, 'SmallBERT_8L', ds_name, seed=SEED)

    ref_embs = extract_embeddings(model, eval_data)
    for pert_name, pset in perturbed_sets.items():
        pert_embs = extract_embeddings(model, pset.sequences)
        report = harness.evaluate('SmallBERT_8L', ref_embs, pert_embs, pert_name)
        row = {
            'architecture': 'SmallBERT_8L',
            'config_name': 'SmallBERT (reference)',
            'n_attn_layers': N_LAYERS,
            'attn_fraction': 1.0,
            'dataset': ds_name,
            'perturbation': pert_name,
            'rdm_similarity': report.rdm_similarity_score,
            'perturbation_stability': report.perturbation_stability_score,
            'composite': report.composite_stability,
            'val_ce': best_val_ce,
            'model_type': 'bert_baseline',
        }
        all_rows.append(row)
    del model
    cleanup_gpu()

# --- Run SmallMamba baseline ---
print(f"\n{'#' * 70}")
print('BASELINE: SmallMamba (pure SSM)')
print('#' * 70)

for ds_name in DATASETS:
    gen_fn = GENERATORS[ds_name]
    train_data, grange = gen_fn(N_TRAIN, seed=SEED)
    eval_data, _ = gen_fn(N_EVAL, seed=SEED + 1000, global_range=grange)
    pert_suite = ContinuousPerturbationSuite(seed=SEED)
    perturbed_sets = pert_suite.run_all(eval_data)

    model = SmallMamba()
    model, t_losses, v_losses, best_val_ce = train_model(
        model, train_data, eval_data, 'SmallMamba_8L', ds_name, seed=SEED)

    ref_embs = extract_embeddings(model, eval_data)
    for pert_name, pset in perturbed_sets.items():
        pert_embs = extract_embeddings(model, pset.sequences)
        report = harness.evaluate('SmallMamba_8L', ref_embs, pert_embs, pert_name)
        row = {
            'architecture': 'SmallMamba_8L',
            'config_name': 'SmallMamba (reference)',
            'n_attn_layers': 0,
            'attn_fraction': 0.0,
            'dataset': ds_name,
            'perturbation': pert_name,
            'rdm_similarity': report.rdm_similarity_score,
            'perturbation_stability': report.perturbation_stability_score,
            'composite': report.composite_stability,
            'val_ce': best_val_ce,
            'model_type': 'mamba_baseline',
        }
        all_rows.append(row)
    del model
    cleanup_gpu()

# Save results
df = pd.DataFrame(all_rows)
csv_path = f'{RESULTS_DIR}/variant_e_attn_ratio_sweep.csv'
df.to_csv(csv_path, index=False)
print(f'\nResults saved to {csv_path}')
print(f'Total rows: {len(df)}')
df.groupby(['config_name', 'dataset'])['composite'].mean()

In [ ]:
# Visualization -- Attention Ratio Sweep Results
#
# Panel A: Dose-Response Curve (Composite Stability vs Attention Fraction)
# Panel B: Dual-Axis (Stability + Val CE vs Attention Fraction)
# Panel C: Per-dataset breakdown

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams.update({'font.size': 11})

# Aggregate by attention fraction
df_sh = df[df['model_type'] == 'striped_hyena']
df_bert = df[df['model_type'] == 'bert_baseline']
df_mamba = df[df['model_type'] == 'mamba_baseline']

agg = df_sh.groupby('attn_fraction').agg(
    comp_mean=('composite', 'mean'),
    comp_std=('composite', 'std'),
    ce_mean=('val_ce', 'mean'),
    ce_std=('val_ce', 'std'),
).reset_index()

bert_comp = df_bert['composite'].mean()
bert_ce = df_bert['val_ce'].mean()
mamba_comp = df_mamba['composite'].mean()
mamba_ce = df_mamba['val_ce'].mean()

fig, axes = plt.subplots(1, 3, figsize=(22, 6))

# ---- Panel A: Attention Ratio Sweep Results
ax = axes[0]
ax.errorbar(agg['attn_fraction'], agg['comp_mean'], yerr=agg['comp_std'],
            marker='o', linewidth=2.5, markersize=10, capsize=5,
            color='#7C3AED', label='StripedHyena (sweep)', zorder=5)

# Reference baselines
ax.axhline(y=mamba_comp, color='#16A34A', linestyle='--', linewidth=2,
           alpha=0.7, label=f'SmallMamba baseline ({mamba_comp:.3f})')
ax.axhline(y=bert_comp, color='#2563EB', linestyle='--', linewidth=2,
           alpha=0.7, label=f'SmallBERT baseline ({bert_comp:.3f})')

# Shade the "tax zone"
ax.fill_between([0, 1], bert_comp, mamba_comp, alpha=0.08,
                color='#EF4444', label='Geometric Tax Zone')

ax.set_xlabel('Attention Fraction (n_attn / n_layers)', fontsize=12)
ax.set_ylabel('Composite Stability', fontsize=12)
ax.set_title('A. Attention Ratio Sweep Results
ax.legend(fontsize=9, loc='best')
ax.set_xlim(-0.05, 1.05)
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))
ax.grid(True, alpha=0.3)

# ---- Panel B: Dual-Axis (Stability + Val CE) ----
ax1 = axes[1]
color_stab = '#7C3AED'
color_ce = '#DC2626'

ln1 = ax1.errorbar(agg['attn_fraction'], agg['comp_mean'],
                    yerr=agg['comp_std'], marker='o', linewidth=2.5,
                    markersize=9, capsize=4, color=color_stab,
                    label='Composite Stability')
ax1.set_xlabel('Attention Fraction', fontsize=12)
ax1.set_ylabel('Composite Stability', color=color_stab, fontsize=12)
ax1.tick_params(axis='y', labelcolor=color_stab)
ax1.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))

ax2 = ax1.twinx()
ln2 = ax2.errorbar(agg['attn_fraction'], agg['ce_mean'],
                    yerr=agg['ce_std'], marker='s', linewidth=2.5,
                    markersize=8, capsize=4, color=color_ce,
                    label='Validation CE')
ax2.set_ylabel('Validation CE', color=color_ce, fontsize=12)
ax2.tick_params(axis='y', labelcolor=color_ce)

lns = [ln1, ln2]
labs = ['Composite Stability', 'Validation CE']
ax1.legend(lns, labs, fontsize=10, loc='center right')
ax1.set_title('B. Stability-Accuracy Tradeoff', fontsize=14,
              fontweight='bold')
ax1.grid(True, alpha=0.3)

# ---- Panel C: Per-dataset breakdown ----
ax = axes[2]
ds_colors = {'waveform': '#3B82F6', 'oscillator': '#F59E0B',
             'lorenz': '#EF4444'}
for ds_name in DATASETS:
    ds_agg = df_sh[df_sh['dataset'] == ds_name].groupby(
        'attn_fraction')['composite'].agg(['mean', 'std']).reset_index()
    ax.errorbar(ds_agg['attn_fraction'], ds_agg['mean'],
                yerr=ds_agg['std'], marker='o', linewidth=2,
                markersize=7, capsize=3, color=ds_colors[ds_name],
                label=ds_name.capitalize())
ax.set_xlabel('Attention Fraction', fontsize=12)
ax.set_ylabel('Composite Stability', fontsize=12)
ax.set_title('C. Per-Dataset Breakdown', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))
ax.grid(True, alpha=0.3)

plt.tight_layout()
fig_path = f'{RESULTS_DIR}/variant_e_dilution_curve.png'
fig.savefig(fig_path, dpi=150, bbox_inches='tight')
fig.savefig(fig_path.replace('.png', '.pdf'), bbox_inches='tight')
plt.show()
print(f'Figure saved to {fig_path}')

In [ ]:
# Phase Transition Detection
#
# Fit piecewise-linear and sigmoid models to the dose-response curve
# to determine whether the relationship is linear, sigmoidal, or
# shows a sharp phase transition.

from scipy.optimize import curve_fit
from scipy.stats import pearsonr, spearmanr

x = agg['attn_fraction'].values
y = agg['comp_mean'].values

# Model 1: Linear
slope, intercept = np.polyfit(x, y, 1)
y_lin = slope * x + intercept
ss_res_lin = np.sum((y - y_lin) ** 2)
ss_tot = np.sum((y - y.mean()) ** 2)
r2_lin = 1 - ss_res_lin / ss_tot if ss_tot > 0 else 0

# Model 2: Sigmoid (logistic)
def sigmoid(x, L, k, x0, b):
    return L / (1 + np.exp(-k * (x - x0))) + b

try:
    popt_sig, _ = curve_fit(sigmoid, x, y,
                             p0=[y.min() - y.max(), 10, 0.5, y.max()],
                             maxfev=10000)
    y_sig = sigmoid(x, *popt_sig)
    ss_res_sig = np.sum((y - y_sig) ** 2)
    r2_sig = 1 - ss_res_sig / ss_tot if ss_tot > 0 else 0
    sig_midpoint = popt_sig[2]
    sig_steepness = abs(popt_sig[1])
except Exception:
    r2_sig = 0
    sig_midpoint = None
    sig_steepness = None

# Monotonicity
rho_spearman, p_spearman = spearmanr(x, y)
r_pearson, p_pearson = pearsonr(x, y)

print('=' * 70)
print('PHASE TRANSITION ANALYSIS')
print('=' * 70)
print(f'\n  Linear fit:   R^2 = {r2_lin:.4f}, slope = {slope:.4f}')
print(f'  Sigmoid fit:  R^2 = {r2_sig:.4f}', end='')
if sig_midpoint is not None:
    print(f', midpoint = {sig_midpoint:.3f}, steepness = {sig_steepness:.2f}')
else:
    print(' (fit failed)')
print(f'\n  Spearman rho: {rho_spearman:.4f} (p = {p_spearman:.2e})')
print(f'  Pearson r:    {r_pearson:.4f} (p = {p_pearson:.2e})')

# Verdict
if r2_sig > r2_lin + 0.05 and sig_steepness is not None and sig_steepness > 5:
    print('\n  VERDICT: PHASE TRANSITION detected (sigmoid >> linear).')
    if sig_midpoint is not None:
        print(f'  Critical attention fraction ~ {sig_midpoint:.1%}')
elif r2_lin > 0.85:
    print('\n  VERDICT: LINEAR dose-response (each attention layer adds a')
    print(f'  fixed quantum of geometric tax: ~{abs(slope)/N_LAYERS:.4f} per layer).')
else:
    print('\n  VERDICT: Nonlinear but not clearly sigmoidal.')
    print('  The curve may reflect diminishing returns or saturation.')

# Largest single-step drop
diffs = np.diff(y)
worst_step = np.argmin(diffs)
print(f'\n  Largest stability drop: {x[worst_step]:.1%} -> {x[worst_step+1]:.1%}')
print(f'    Delta composite: {diffs[worst_step]:.4f}')

In [ ]:
# Summary

print('=' * 70)
print('VARIANT E: ATTENTION RATIO SWEEP -- FINAL SUMMARY')
print('=' * 70)

print('\n--- StripedHyena Sweep ---')
print(f'  {"Config":>20s}  {"Attn%":>6s}  {"Composite":>12s}  {"Val CE":>10s}')
print('  ' + '-' * 56)

for _, row in agg.sort_values('attn_fraction').iterrows():
    frac = row['attn_fraction']
    label = f'{int(frac * N_LAYERS)}/{N_LAYERS}'
    print(f'  {label:>20s}  {frac:>5.1%}  '
          f'{row["comp_mean"]:>8.4f}+/-{row["comp_std"]:.4f}  '
          f'{row["ce_mean"]:>10.4f}')

print(f'\n--- Reference Baselines ({N_LAYERS} layers) ---')
print(f'  SmallBERT:  composite={bert_comp:.4f}, val_CE={bert_ce:.4f}')
print(f'  SmallMamba: composite={mamba_comp:.4f}, val_CE={mamba_ce:.4f}')

# Tax per attention layer
pure_hyena_comp = agg[agg['attn_fraction'] == 0]['comp_mean'].values[0]
pure_attn_comp = agg[agg['attn_fraction'] == 1.0]['comp_mean'].values
if len(pure_attn_comp) > 0:
    total_tax = pure_hyena_comp - pure_attn_comp[0]
    print(f'\n  Total geometric tax (0% -> 100% attn): {total_tax:.4f}')
    print(f'  Average tax per attention layer: {total_tax / N_LAYERS:.4f}')